# Core 05 - System (CLI)

Notebook CLI paralelo a `tutorials/core/05_system.ipynb`.

**Objetivo:** Compilar y ejecutar un System.

Este notebook no llama factories de Agentic Systems directamente: ejecuta el
entrypoint CLI real, conserva la salida Rich y valida después el JSON del mismo
contrato.


## Cómo se ejecuta

La forma portable es `python -m agentic_systems.cli ...`. Después de instalar
el wheel, el entrypoint equivalente es `agentic-systems ...`.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def _repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio.")


ROOT = _repo_root()
CLI = [sys.executable, "-m", "agentic_systems.cli"]


def run_cli(*args: str, expected: int = 0) -> str:
    env = os.environ.copy()
    source_path = str(ROOT / "src")
    env["PYTHONPATH"] = (
        source_path
        if not env.get("PYTHONPATH")
        else source_path + os.pathsep + env["PYTHONPATH"]
    )
    command = [*CLI, *args]
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        check=False,
    )
    print("$ " + " ".join(command))
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    assert completed.returncode == expected, completed.stderr
    return completed.stdout


def run_cli_json(*args: str) -> dict:
    return json.loads(run_cli(*args, "--json"))


def assert_rich(output: str, title: str) -> None:
    assert title in output
    ascii_box = "+" in output and "|" in output
    unicode_box = "─" in output and "│" in output
    assert ascii_box or unicode_box


## 1) Salida humana Rich

La celda conserva stdout y comprueba título y bordes. Esto detecta tablas o
paneles truncados, además del exit code.


In [2]:
rich_output = run_cli(*['system', 'run', '--value', 'cli'])
assert_rich(rich_output, 'System Workflow')


$ C:\Python314\python.exe -m agentic_systems.cli system run --value cli
+------------------------------ System Workflow ------------------------------+
| {                                                                           |
|   "result": {                                                               |
|     "data": {                                                               |
|       "ok": true,                                                           |
|       "tool": "cli_echo",                                                   |
|       "value": "cli"                                                        |
|     },                                                                      |
|     "engine": "agentic-system",                                             |
|     "final": {                                                              |
|       "ok": true,                                                           |
|       "tool": "cli_echo",                     

## 2) Contrato de máquina

La misma ruta se ejecuta con `--json` para afirmar campos y cardinalidad sin
parsear la presentación Rich.


In [3]:
payload = run_cli_json(*['system', 'run', '--value', 'cli'])
assert payload["result"]["ok"] is True
payload


$ C:\Python314\python.exe -m agentic_systems.cli system run --value cli --json
{
  "result": {
    "data": {
      "ok": true,
      "tool": "cli_echo",
      "value": "cli"
    },
    "engine": "agentic-system",
    "final": {
      "ok": true,
      "tool": "cli_echo",
      "value": "cli"
    },
    "mode": "eval",
    "ok": true,
    "text": "cli_echo -> {\"value\": \"cli\"}",
    "tool_outputs": [
      {
        "input": {
          "value": "cli"
        },
        "ok": true,
        "output": {
          "value": "cli"
        },
        "tool": "cli_echo"
      }
    ],
    "tools_called": [
      "cli_echo"
    ],
    "usage": {},
    "validation_ok": null
  },
  "scenario": "system",
  "scenario_api_ids": [
    "system",
    "AgenticSystem.run",
    "RunResult"
  ],
  "workflow": "system"
}


{'result': {'data': {'ok': True, 'tool': 'cli_echo', 'value': 'cli'},
  'engine': 'agentic-system',
  'final': {'ok': True, 'tool': 'cli_echo', 'value': 'cli'},
  'mode': 'eval',
  'ok': True,
  'text': 'cli_echo -> {"value": "cli"}',
  'tool_outputs': [{'input': {'value': 'cli'},
    'ok': True,
    'output': {'value': 'cli'},
    'tool': 'cli_echo'}],
  'tools_called': ['cli_echo'],
  'usage': {},
  'validation_ok': None},
 'scenario': 'system',
 'scenario_api_ids': ['system', 'AgenticSystem.run', 'RunResult'],
 'workflow': 'system'}

## Resultado e interpretación

Rich responde a lectura humana; JSON responde a automatización. Ambos nacen del
mismo comando y del mismo escenario público. Un estado `not-run` conserva el
motivo, pero no cuenta como evidencia live.
